<a href="https://colab.research.google.com/github/codebysumit/cryptography-algorithms/blob/master/notebooks/one_time_pad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# One-Time Pad (OTP)

## History
The One-Time Pad was developed in the early 1900s. It is credited to Frank Miller in 1882 for telegraph use, and later to Gilbert Vernam and Joseph Mauborgne around 1917-1919, who built it into a practical system for military and diplomatic communication. It was used heavily during the Cold War, including on the famous Moscow-Washington hotline. Unlike every other cipher in this course, the One-Time Pad has been mathematically proven to be unbreakable, as long as it is used correctly.

## What is One-Time Pad?
The One-Time Pad works on the same basic idea as the Vigenere Cipher: add a key value to a plaintext value, character by character, using modular arithmetic. The big difference is in the key itself. In Vigenere, a short keyword is repeated again and again to cover the whole message. In One-Time Pad, there is no repeating.

For a One-Time Pad to give perfect security, three rules must all be followed:

1.  **The key must be truly random.** Not a password, not a sentence, not a pattern, but random noise with no structure at all.
2.  **The key must be at least as long as the message.** Every single character of the plaintext needs its own fresh key character.
3.  **The key must never be reused.** Once a key is used for one message, it must be destroyed and never used again for any other message.

If even one of these three rules is broken, the cipher stops being unbreakable. A repeated key turns the One-Time Pad straight back into a Vigenere Cipher, which we already showed can be broken with frequency analysis.

In this implementation, we use the printable ASCII range from space (` `) to tilde (`~`), which spans from ASCII value 32 to 126 ($N=95$).

## Cryptography Algorithm

### Constants
*   $S = 32$ (Start of printable ASCII range)
*   $E = 126$ (End of printable ASCII range)
*   $N = E - S + 1 = 95$ (Total number of printable characters)
*   $K$ = The one-time pad key, a truly random string, with the same length as the plaintext
*   $x$ = Numeric value of the plaintext character ($0 \le x < N$)
*   $y$ = Numeric value of the ciphertext character ($0 \le y < N$)
*   $k_i$ = Numeric value of the key character at position $i$ ($0 \le k_i < N$)

### 1. Key Generation
A fresh random key is generated for every single message, with a length exactly equal to the plaintext length. It is never derived from a word, a phrase, or anything memorable, because anything memorable has a pattern, and a pattern is a weakness.

### 2. Encryption
For each character in the plaintext, the character index $x$ at position $i$ is transformed into a ciphertext index $y$ using the formula:

$$E(x_i) = (x_i + k_i) \pmod N$$

To get the final ASCII value: $C = E(x_i) + S$

### 3. Decryption
The decryption function subtracts the same key value that was used for encryption at that position:

$$D(y_i) = (y_i - k_i) \pmod N$$

To get the final ASCII value: $P = D(y_i) + S$

### Why It Is Unbreakable
Because the key is truly random and exactly as long as the message, every possible plaintext of that length is equally likely to have produced the observed ciphertext. There is no statistical pattern left in the ciphertext for an attacker to latch onto, because the key never repeats and carries no structure of its own. This is very different from the Vigenere Cipher, where a short repeating key eventually leaks its own pattern into the ciphertext.

### Key Requirements and Practical Limitations
*   The key must be **truly random**, generated by a proper random source, not a normal pseudo-random generator used carelessly.
*   The key must be **as long as the message**. A long message needs an equally long key, which is hard to generate, store, and share safely.
*   The key must be **used only once**, then destroyed. This is where the name One-Time Pad comes from.
*   Because of these strict requirements, One-Time Pad is rarely used in everyday systems. It is mostly seen in very high security, low volume communication, where the cost of managing huge random keys is acceptable.

### 1. Import Dependencies

In [1]:
import random

### 2. Helper Utilities

In [2]:
def build_key_stream(text_length: int, key: str) -> str:
    # in a true One-Time Pad the key is already the same length as the plaintext
    # this check makes sure that rule is never broken by mistake
    if len(key) < text_length:
        raise ValueError("'key' must be at least as long as the plaintext for a true One-Time Pad.")
    return key[:text_length]

### 3. Generate a Truly Random Key

In [3]:
def generate_random_key(length: int) -> str:
    START_ASCII = 32
    END_ASCII = 126

    # random.SystemRandom pulls randomness from the operating system
    # this is much closer to true randomness than the default random module
    secure_random = random.SystemRandom()
    return "".join(chr(secure_random.randint(START_ASCII, END_ASCII)) for _ in range(length))

### 4. Encryption

In [4]:
def encrypt(text: str, key: str) -> str:
    START_ASCII = 32
    END_ASCII = 126
    TOTAL_CHAR = END_ASCII - START_ASCII + 1  # 95 characters

    key_stream = build_key_stream(len(text), key)
    encrypted_text = ""

    for i, ch in enumerate(text):
        code = ord(ch)
        if START_ASCII <= code <= END_ASCII:
            shift = ord(key_stream[i]) - START_ASCII
            cipher_code = ((code - START_ASCII) + shift) % TOTAL_CHAR
            encrypted_text += chr(cipher_code + START_ASCII)
        else:
            encrypted_text += ch

    return encrypted_text

### 5. Decryption

In [5]:
def decrypt(text: str, key: str) -> str:
    START_ASCII = 32
    END_ASCII = 126
    TOTAL_CHAR = END_ASCII - START_ASCII + 1  # 95 characters

    key_stream = build_key_stream(len(text), key)
    decrypted_text = ""

    for i, ch in enumerate(text):
        code = ord(ch)
        if START_ASCII <= code <= END_ASCII:
            shift = ord(key_stream[i]) - START_ASCII
            plain_code = ((code - START_ASCII) - shift) % TOTAL_CHAR
            decrypted_text += chr(plain_code + START_ASCII)
        else:
            decrypted_text += ch

    return decrypted_text

### 6. Example usage

In [7]:
plaintext = """TOP secret Massage! Agent 101, visit Area 51 (37d14'0\"N 115d48'30\"W)."""

key = generate_random_key(len(plaintext))
print(f"Generated One-Time Pad Key (same length as message): {key}")

Generated One-Time Pad Key (same length as message): fGnrd7Z;Wbm3X1mPS +EdfZ3h/1rVy=VFt~C17sPpZmIRV%zbg`H&<L[gCAO+oe"OF8MJ


In [8]:
print(f"Original Plain Text: {plaintext}")

cipher_text = encrypt(plaintext, key)
print("Encrypted:", cipher_text)

decrypted_text = decrypt(cipher_text, key)
print("Decrypted:", decrypted_text)

match = plaintext == decrypted_text
print(f"Verification Match:{match}")

Original Plain Text: TOP secret Massage! Agent 101, visit Area 51 (37d14'0"N 115d48'30"W).
Encrypted: ;v?rX|>.=Wm`:%a2;e,E&N@"]/B#g&=M0hh81Xf6RZ#ZR^82GxtO6>z[xTV4?(l5_HoVX
Decrypted: TOP secret Massage! Agent 101, visit Area 51 (37d14'0"N 115d48'30"W).
Verification Match:True


### 7. Important Reminder: Never Reuse a Key

The cell below is only here to show what NOT to do. It encrypts two different messages using the exact same key, which completely breaks the security guarantee of a One-Time Pad. This is called a **two-time pad** mistake, and it is a very common real world failure.

In [9]:
message_one = "MEET ME AT THE OLD BRIDGE TONIGHT"
message_two = "DO NOT TRUST THE COURIER THIS TIME"

# WRONG: reusing the same key for two different messages
reused_key = generate_random_key(max(len(message_one), len(message_two)))

cipher_one = encrypt(message_one, reused_key)
cipher_two = encrypt(message_two, reused_key)

print("Cipher One:", cipher_one)
print("Cipher Two:", cipher_two)
print("\nBecause the same key was used twice, an attacker who studies both")
print("ciphertexts together can start pulling out patterns between the two")
print("messages, even without ever knowing the key. This is exactly why")
print("rule 3 (never reuse a key) is non negotiable in real use.")

Cipher One: z;h.C#Z6.(+AFEo>EQROCX\jy]qt'UA"j
Cipher Two: qEC(r*5j?)^A}T84xP"bCX]uT2en,,N#cY

Because the same key was used twice, an attacker who studies both
ciphertexts together can start pulling out patterns between the two
messages, even without ever knowing the key. This is exactly why
rule 3 (never reuse a key) is non negotiable in real use.
